# CA3 HW3: Energy-Based and Score-Based Models on MNIST

**Objectives**

- Implement EBM with Langevin sampling and contrastive divergence.
- Implement NCSN with weighted DSM and annealed Langevin dynamics (unconditional + conditional).
- Provide training, sampling, and denoising pipelines with reproducibility hooks.

**Structure**

1. Setup and configuration
2. Data loading and visualization
3. EBM model, training, sampling, denoising
4. NCSN model, training, sampling (ALD), denoising, conditional variant
5. Results logging placeholders
6. Reproducibility notes


In [ ]:
# Setup and Configuration
import os, random
import numpy as np
import torch
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_ROOT = PROJECT_ROOT / "data" / "mnist"


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# Configs
from next_year.CA3.codes.config import DataConfig, EBMConfig, NCSNConfig, RunPaths

data_cfg = DataConfig()
ebm_cfg = EBMConfig(device=device)
ncsn_cfg = NCSNConfig(device=device)
paths = RunPaths()
paths.ensure()
data_cfg, ebm_cfg, ncsn_cfg

In [ ]:
# Data loading and a quick peek
from next_year.CA3.codes.data import mnist_dataloaders
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

train_loader, test_loader = mnist_dataloaders(data_cfg, normalize_to_minus1_1=False)
images, labels = next(iter(train_loader))
grid = make_grid(images[:16], nrow=4)
plt.figure(figsize=(4, 4))
plt.axis("off")
plt.imshow(grid.permute(1, 2, 0))
plt.show()

In [ ]:
# EBM model, sampler, and a short training utility (configurable)
from next_year.CA3.codes.ebm_model import ConvEnergyModel
from next_year.CA3.codes.ebm_sampling import LangevinSampler, sample_from_noise
from torch import optim
from tqdm import tqdm


def train_ebm(train_loader, test_loader, cfg: EBMConfig, epochs: int = 1):
    model = ConvEnergyModel().to(cfg.device)
    sampler = LangevinSampler(model, cfg)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    history = []
    for epoch in range(1, epochs + 1):
        loop = tqdm(train_loader, desc=f"EBM epoch {epoch}/{epochs}", leave=False)
        for x, _ in loop:
            x = x.to(cfg.device)
            x_fake = sampler(torch.rand_like(x))
            e_real = model(x)
            e_fake = model(x_fake)
            data_term = e_real.mean() - e_fake.mean()
            reg_term = cfg.lambda_reg * (e_real.pow(2).mean() + e_fake.pow(2).mean())
            loss = data_term + reg_term
            opt.zero_grad()
            loss.backward()
            opt.step()
            history.append(loss.item())
        with torch.no_grad():
            samples = sample_from_noise(model, cfg, (16, 1, 28, 28))
            grid = make_grid(samples, nrow=4, normalize=True)
            plt.figure(figsize=(4, 4))
            plt.axis("off")
            plt.imshow(grid.permute(1, 2, 0))
            plt.show()
    return model, history


# Note: For full training, run ebm_train.py from the module; this cell is for interactive/smoke usage.

In [ ]:
# NCSN model, DSM loss, and ALD sampling utilities
from next_year.CA3.codes.ncsn_model import ScoreNet
from next_year.CA3.codes.ncsn_loss import dsm_loss
from next_year.CA3.codes.ncsn_sampling import sample as ncsn_sample


def train_ncsn(
    train_loader, cfg: NCSNConfig, epochs: int = 1, conditional: bool = False
):
    cfg.conditional = conditional
    model = ScoreNet(cfg).to(cfg.device)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    sigmas = cfg.sigmas
    history = []
    loop_epochs = range(1, epochs + 1)
    for epoch in loop_epochs:
        loop = tqdm(train_loader, desc=f"NCSN epoch {epoch}/{epochs}", leave=False)
        for x, y in loop:
            x = x.to(cfg.device) * 2 - 1
            y_lbl = y.to(cfg.device) if conditional else None
            loss = dsm_loss(model, x, cfg, sigmas, y_lbl)
            opt.zero_grad()
            loss.backward()
            opt.step()
            history.append(loss.item())
        with torch.no_grad():
            y_samples = (
                (torch.arange(0, 16, device=cfg.device) % cfg.num_classes)
                if conditional
                else None
            )
            samples = ncsn_sample(model, cfg, num_samples=16, y=y_samples)
            grid = make_grid(
                (samples + 1) / 2, nrow=4, normalize=True, value_range=(0, 1)
            )
            plt.figure(figsize=(4, 4))
            plt.axis("off")
            plt.imshow(grid.permute(1, 2, 0))
            plt.show()
    return model, history


# Note: For full training, use ncsn_train.py entrypoint. This cell is for interactive smoke tests or short runs.

## Usage Notes

- For full runs and saving artifacts, prefer the script entrypoints: `python -m next_year.CA3.codes.ebm_train` and `python -m next_year.CA3.codes.ncsn_train`.
- This notebook provides compact, interactive-friendly versions for exploration or short sanity checks (set epochs small).
- Ensure `torch` and `torchvision` are installed (see `requirements.txt`).
- Figures shown inline are not automatically saved; scripts handle saving to `images/`.


## Quick demo run (short, inline)

- Runs 1 epoch EBM and NCSN (unconditional) with default configs.
- Uses small epochs to keep runtime manageable; for full quality use the scripts.
- Displays sample grids inline (not saved); ensure `torch` is installed and GPU is recommended.


In [ ]:
# Demo: run short EBM and NCSN training and show samples
# WARNING: still compute-intensive; adjust epochs/steps if needed.

# EBM short run (1 epoch)
short_ebm_cfg = ebm_cfg
short_ebm_cfg.langevin_steps = 20  # faster demo
short_ebm_cfg.sample_grid = 8
model_ebm, ebm_hist = train_ebm(train_loader, test_loader, short_ebm_cfg, epochs=1)

# NCSN short run (1 epoch, unconditional)
short_ncsn_cfg = ncsn_cfg
short_ncsn_cfg.num_levels = 5  # faster demo
short_ncsn_cfg.langevin_steps = 50
model_ncsn, ncsn_hist = train_ncsn(
    train_loader, short_ncsn_cfg, epochs=1, conditional=False
)

print(f"EBM steps: {len(ebm_hist)} losses logged")
print(f"NCSN steps: {len(ncsn_hist)} losses logged")

## Visualize demo losses

Plot the loss histories from the short demo runs to quickly inspect optimization behavior.


In [ ]:
import matplotlib.pyplot as plt

if "ebm_hist" in locals() and ebm_hist:
    plt.figure(figsize=(6, 4))
    plt.plot(ebm_hist)
    plt.title("EBM Loss (demo)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.show()
else:
    print("Run the demo cell to populate ebm_hist.")

if "ncsn_hist" in locals() and ncsn_hist:
    plt.figure(figsize=(6, 4))
    plt.plot(ncsn_hist)
    plt.title("NCSN DSM Loss (demo)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.show()
else:
    print("Run the demo cell to populate ncsn_hist.")

## Display saved figures if available

Checks for images produced by the script entrypoints (e.g., `images/ebm/ebm_samples_epoch10.png`) and shows them inline when present.


In [ ]:
from matplotlib import image as mpimg

candidates = [
    paths.images / "ebm" / "ebm_samples_epoch10.png",
    paths.images / "ebm" / "ebm_denoised_epoch10.png",
    paths.images / "ncsn" / "samples_epoch30.png",
    paths.images / "ncsn_cond" / "samples_epoch30.png",
    paths.images / "ncsn_infer" / "denoised_0.40.png",
]

for img_path in candidates:
    if img_path.exists():
        img = mpimg.imread(img_path)
        plt.figure(figsize=(5, 5))
        plt.axis("off")
        plt.title(img_path.name)
        plt.imshow(img)
        plt.show()
    else:
        print(f"Missing: {img_path}")